# 01 · Parse the drawing PDFs with `ai_parse_document`

First step of the document-intelligence chain (parse, then extract and classify, then check compliance). It reads the 1,200 drawing PDFs that the drawings pipeline loaded into `bluerridge_bronze_drawings.raw_files` (the file bytes are in `content`), runs the built-in `ai_parse_document` AI function on each one, and writes one row per drawing to `bluerridge_silver_engineering.drawings_parsed`:

| column | type | what it holds |
|---|---|---|
| `drawing_id` | STRING | the file name without `.pdf`; joins to `drawings_raw_tracking` |
| `volume_path` | STRING | the `/Volumes/...` path of the PDF |
| `parsed_markdown` | STRING | the drawing's text in reading order, for `ai_extract` or an LLM prompt: titles and section headers as headings, the schematic's labels in a `figure` block, tables as HTML, and `<!-- page_num: N -->` where a page starts (0-based, like the ground truth) |
| `parse_status` | STRING | `ok`, or `error:<reason>` |
| `parse_json` | STRING | the function's whole output as JSON: the pages, and for each element its type, text, confidence, bounding box in pixels and (for figures) a generated description. The compliance judge reads page numbers and boxes from here |
| `parsed_at` | TIMESTAMP | when the table was written |

How it runs:

- Files whose `_rescued_data` is set (not a real PDF, see the drawings pipeline) are skipped.
- One `ai_parse_document(content)` call per document, written as a Spark expression, so the calls spread over the compute instead of running in a Python loop. The function takes no model argument (checked below), so the service's default model does the parsing.
- The AI step runs once, in the write cell. Everything after it reads the finished table, so no drawing is parsed twice. Running the notebook again parses all 1,200 drawings again: about 1,370 pages, which the published AI Functions price list puts at roughly $6 per 1,000 pages for engineering diagrams (a directional estimate), so under $10.
- Parsing is done by a model, so it is not repeatable to the letter. Parsing one PDF twice can return the same labels in a different order, boxes a few pixels apart, or a tag read slightly differently (`LOTO` or `LOT0`). A re-run therefore changes the table a little.

In [1]:
import contextlib
import io
import json
import re
import time
from pathlib import Path

from pyspark.sql import functions as F

# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
# 0 parses every drawing. A positive number parses only that many (a fixed pseudo-random subset): the cost cap.
dbutils.widgets.text("max_rows", "0", "Max drawings to parse (0 = all)")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")
MAX_ROWS = int(dbutils.widgets.get("max_rows") or 0)

BRONZE = f"{catalog}.{prefix}_bronze_drawings.raw_files"
TRACKING = f"{catalog}.{prefix}_silver_engineering.drawings_raw_tracking"
TARGET = f"{catalog}.{prefix}_silver_engineering.drawings_parsed"

PARTITIONS = 32  # tasks that make the AI calls side by side
SAMPLE_ID = "BRL-INS-PID-0052"  # written out in full at the end: a P&ID with three planted flaws
OUT = Path("outputs")  # next to this notebook

print(f"source: {BRONZE}")
print(f"ids from: {TRACKING}")
print(f"target: {TARGET}")
print(f"drawings to parse: {MAX_ROWS or 'all'}")

source: mfg_mc_se_sa.bluerridge_bronze_drawings.raw_files
ids from: mfg_mc_se_sa.bluerridge_silver_engineering.drawings_raw_tracking
target: mfg_mc_se_sa.bluerridge_silver_engineering.drawings_parsed
drawings to parse: all


## Which call signature works?

The plan prefers a cheaper model if `ai_parse_document` accepts one, so each form is tried on a single drawing. A bad argument is refused as the query starts, before the model sees anything; the two valid forms each parse that one drawing.

In [2]:
def accepted(call):
    # (True, "") if `SELECT <call>` runs on one drawing, else (False, the error's first line).
    try:
        spark.sql(f"SELECT CAST({call} AS STRING) FROM {BRONZE} WHERE _rescued_data IS NULL LIMIT 1").collect()
        return True, ""
    except Exception as e:
        return False, str(e).strip().splitlines()[0][:230]


for call in (
    "ai_parse_document(content)",
    "ai_parse_document(content, map('version', '2.0'))",
    "ai_parse_document(content, model => 'databricks-claude-haiku-4-5')",
    "ai_parse_document(content, 'databricks-claude-haiku-4-5')",
):
    ok, why = accepted(call)
    print(f"{'accepted' if ok else 'rejected'}  {call}")
    if not ok:
        print(f"          {why}")

accepted  ai_parse_document(content)
accepted  ai_parse_document(content, map('version', '2.0'))
rejected  ai_parse_document(content, model => 'databricks-claude-haiku-4-5')
          [AI_FUNCTION_COMPILATION_ERROR] An error occurred while compiling the AI function: INVALID_ARGUMENT: Unrecognized parameter name `model` for function `ai_parse_document` SQLSTATE: 42883; line 1 pos 12
rejected  ai_parse_document(content, 'databricks-claude-haiku-4-5')
          [AI_FUNCTION_COMPILATION_ERROR] An error occurred while compiling the AI function: INVALID_ARGUMENT: Call to `ai_parse_document` is invalid because options requires MAP<STRING,STRING>, but got STRING SQLSTATE: 42883; line 1 pos 12


Only the second argument can be set, and it is an options map (`version`, `imageOutputPath`, `descriptionElementTypes`, `pageRange`), not a model. So there is no cheaper model to choose and the notebook uses the one-argument form with the service default.

## Pick the documents

Bronze rows with `_rescued_data IS NULL` are real PDFs. `drawing_id` and `volume_path` come from `drawings_raw_tracking`, joined on the source file, so the key is the one the rest of the silver layer uses.

In [3]:
bronze = spark.table(BRONZE).where("_rescued_data IS NULL").select("_source_file", "content")
tracking = spark.table(TRACKING).select("drawing_id", "volume_path", "_source_file")
docs = bronze.join(tracking, "_source_file").select("drawing_id", "volume_path", "content")

n_bronze = spark.table(BRONZE).count()
n_pdf = bronze.count()
n_docs = docs.count()
print(f"bronze rows: {n_bronze}; real PDFs (_rescued_data IS NULL): {n_pdf}; of those, found in drawings_raw_tracking: {n_docs}")
assert n_docs == n_pdf, "some PDFs have no row in drawings_raw_tracking"

if MAX_ROWS:
    # A fixed pseudo-random subset (the same on every run) that always includes the drawing written out in full below.
    docs = docs.orderBy(F.col("drawing_id") != SAMPLE_ID, F.xxhash64("drawing_id")).limit(MAX_ROWS)
    n_docs = min(MAX_ROWS, n_docs)
print(f"drawings to parse: {n_docs}")

bronze rows: 1200; real PDFs (_rescued_data IS NULL): 1200; of those, found in drawings_raw_tracking: 1200
drawings to parse: 1200


## Turn the result into the table's columns

`ai_parse_document` returns a VARIANT. The notebook casts it to a JSON string inside the SQL expression, so the table holds plain strings and the Python side never handles a VARIANT, and derives the other two columns from that string:

- `parsed_markdown`: the elements in the order the parser returned them. A `title` becomes `# heading`, a `section_header` `## heading`, a `figure` (the schematic's labels, as text) a fenced `figure` block, a `table` stays the HTML the parser produced, everything else is a paragraph. Elements without text are left out here (they stay in `parse_json`), and so are the generated figure descriptions.
- `parse_status`: `ok`, or `error:<reason>` when the function reports an `error_status`, returns nothing, or finds no text at all.

The two functions below never raise, because one bad row must not fail a run that has already paid for 1,200 parses. They are checked on hand-made results first, including the error cases that real data may not produce.

In [4]:
def to_markdown(parse_json):
    # Reading-order markdown from the function's JSON, or None when there is no text.
    try:
        elements = (json.loads(parse_json).get("document") or {}).get("elements") or []
        blocks, page = [], None
        for el in elements:
            text = (el.get("content") or "").strip()
            if not text:
                continue
            boxes = el.get("bbox") or []
            if boxes and boxes[0].get("page_id") not in (None, page):
                page = boxes[0]["page_id"]
                blocks.append(f"<!-- page_num: {page} -->")
            kind = el.get("type")
            if kind == "title":
                blocks.append("# " + " ".join(text.split()))
            elif kind == "section_header":
                blocks.append("## " + " ".join(text.split()))
            elif kind == "figure":
                blocks.append(f"```figure\n{text}\n```")
            else:
                blocks.append(text)
        return "\n\n".join(blocks) or None
    except Exception:
        return None


def parse_status(parse_json):
    # "ok", or "error:<reason>".
    if not parse_json:
        return "error:no result"
    try:
        result = json.loads(parse_json)
        errors = result.get("error_status")
        if errors:
            if isinstance(errors, list):
                errors = "; ".join(f"page {e.get('page_id')}: {e.get('error_message')}" for e in errors)
            return f"error:{errors}"[:300]
        elements = (result.get("document") or {}).get("elements") or []
        if not any((el.get("content") or "").strip() for el in elements):
            return "error:no text found"
        return "ok"
    except Exception as e:
        return f"error:unreadable result ({type(e).__name__})"


markdown_udf = F.udf(to_markdown, "string")
status_udf = F.udf(parse_status, "string")

# Hand-made results: a good one, a page error, no text, no result, and something that is not JSON.
good = json.dumps({"document": {"pages": [{"id": 0}], "elements": [
    {"type": "title", "content": "SITE PLAN", "bbox": [{"coord": [1, 2, 3, 4], "page_id": 0}]},
    {"type": "text", "content": "NOTE 1", "bbox": [{"coord": [1, 2, 3, 4], "page_id": 1}]},
    {"type": "figure", "content": "", "bbox": [{"coord": [1, 2, 3, 4], "page_id": 1}]}]}, "error_status": None})
failed = json.dumps({"document": {"pages": [], "elements": []}, "error_status": [{"error_message": "could not render page", "page_id": 3}]})
empty = json.dumps({"document": {"pages": [{"id": 0}], "elements": []}, "error_status": None})
for name, value in (("good", good), ("page error", failed), ("no text", empty), ("None", None), ("not JSON", "<html>")):
    print(f"{name:10} -> {parse_status(value)!r}, markdown {to_markdown(value)!r}")
assert parse_status(good) == "ok"
assert to_markdown(good) == "<!-- page_num: 0 -->\n\n# SITE PLAN\n\n<!-- page_num: 1 -->\n\nNOTE 1"
assert parse_status(failed) == "error:page 3: could not render page"

good       -> 'ok', markdown '<!-- page_num: 0 -->\n\n# SITE PLAN\n\n<!-- page_num: 1 -->\n\nNOTE 1'
page error -> 'error:page 3: could not render page', markdown None
no text    -> 'error:no text found', markdown None
None       -> 'error:no result', markdown None
not JSON   -> 'error:unreadable result (JSONDecodeError)', markdown None


## Build the DataFrame

Nothing runs yet: a DataFrame is a plan. `repartition` spreads the documents over several tasks, because a table this small would otherwise be read as one or two partitions and the AI calls would run in a line. The plan is checked for the cost guard before anything is written: it must hold exactly one `AI Query` step, even though three columns are built from its result.

In [5]:
parsed = (
    docs.repartition(PARTITIONS)
    .withColumn("parse_json", F.expr("CAST(ai_parse_document(content) AS STRING)"))
    .select(
        "drawing_id",
        "volume_path",
        markdown_udf("parse_json").alias("parsed_markdown"),
        status_udf("parse_json").alias("parse_status"),
        "parse_json",
        F.current_timestamp().alias("parsed_at"),
    )
)
print("columns:", parsed.columns)

# explain() only plans the query; it does not run it.
plan = io.StringIO()
with contextlib.redirect_stdout(plan):
    parsed.explain(mode="formatted")
ai_steps = re.findall(r"^\(\d+\) AI Query.*$", plan.getvalue(), flags=re.M)
print(f"AI Query steps in the physical plan: {len(ai_steps)}")
assert len(ai_steps) == 1, "ai_parse_document would run more than once per document"

columns: ['drawing_id', 'volume_path', 'parsed_markdown', 'parse_status', 'parse_json', 'parsed_at']
AI Query steps in the physical plan: 1


## Parse and write

The only cell that calls the AI function. `overwrite` makes a re-run replace the table instead of appending a second copy of every drawing.

In [6]:
t0 = time.time()
parsed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
print(f"parsed {n_docs} drawings and wrote {TARGET} in {time.time() - t0:.0f} s")

parsed 1200 drawings and wrote mfg_mc_se_sa.bluerridge_silver_engineering.drawings_parsed in 632 s


Document the table in Unity Catalog (a table comment, column comments, `quality = silver` like the other silver tables) and show the schema it ended up with:

In [7]:
spark.sql(
    f"COMMENT ON TABLE {TARGET} IS 'One row per drawing PDF, parsed by ai_parse_document (service default model). "
    "parsed_markdown is the text in reading order, parse_json the full result.'"
)
COLUMN_COMMENTS = {
    "drawing_id": "File name without .pdf; key into drawings_raw_tracking",
    "volume_path": "Unity Catalog Volume path of the PDF",
    "parsed_markdown": "Drawing text in reading order; <!-- page_num: N --> marks the start of page N (0-based)",
    "parse_status": "ok, or error:<reason>",
    "parse_json": "Full ai_parse_document output as JSON: pages, and elements with type, content, confidence, bbox (pixels) and description",
    "parsed_at": "When the table was written",
}
for column, comment in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN {column} COMMENT '{comment}'")
spark.sql(f"ALTER TABLE {TARGET} SET TBLPROPERTIES ('quality' = 'silver')")

display(spark.sql(f"""
    SELECT ordinal_position AS position, column_name, data_type, comment
    FROM {catalog}.information_schema.columns
    WHERE table_schema = '{prefix}_silver_engineering' AND table_name = 'drawings_parsed'
    ORDER BY ordinal_position
"""))

position,column_name,data_type,comment
0,drawing_id,STRING,File name without .pdf; key into drawings_raw_tracking
1,volume_path,STRING,Unity Catalog Volume path of the PDF
2,parsed_markdown,STRING,Drawing text in reading order; <!-- page_num: N --> marks the start of page N (0-based)
3,parse_status,STRING,"ok, or error:<reason>"
4,parse_json,STRING,"Full ai_parse_document output as JSON: pages, and elements with type, content, confidence, bbox (pixels) and description"
5,parsed_at,TIMESTAMP,When the table was written


## Read it back

From here on the notebook reads the table, never the DataFrame above, so nothing is parsed again. Five random drawings with the first 500 characters of their text (whitespace collapsed so each row stays on one line; the table holds the full text):

In [8]:
display(
    spark.table(TARGET)
    .orderBy(F.rand(seed=42))
    .limit(5)
    .select("drawing_id", F.regexp_replace(F.substring("parsed_markdown", 1, 500), r"\s+", " ").alias("preview"))
)

drawing_id,preview
BRL-INS-PID-0370,"<!-- page_num: 0 --> # PROCESS FLOW AND INSTRUMENTATION - PIPE RACK SOUTH ```figure PSV-2601 SET 475 PSIG PSV-2603 SET 475 PSIG FIC 2660 M LOT0-2602 FROM FEED GAS HEADER FLG CL300 FLG CL300 HV-2671 TO LOADING LINE 16""-NG-2650-B3S DP 475 PSIG 16""-NG-2650-B3S DP 475 PSIG K-2602 BOG COMPRESSOR V-2601 SEPARATOR DP 475 PSIG / -270 TO 140 F V-2603 KNOCKOUT DRUM DP 475 PSIG / -270 TO 140 F ``` ```figure PSV-2606 SET 645 PSIG M LOT0-2604 FIC 2661 M LOT0-2605 FROM FEED GAS HEADER FLG CL300 FLG CL300 H"
BRL-STR-GA-0736,"<!-- page_num: 0 --> ```figure 96'-8"" A EXIT 3 CLR 48 IN B C D E 4 CONTROL BUILDING - PLAN FINISHED FLOOR EL. 100'-0"" CONTROL ROOM O LOTO-3203 K-3203 BOG COMPRESSOR O LOTO-3202 LCP-3202 LOCAL PANEL 3 2 O LOTO-3204A P-3204A BOOSTER PUMP O LOTO-3201A P-3201A LNG PUMP EXIT 1 CLR 40 IN ELECTRICAL / MCC ROOM EXIT 2 CLR 56 IN 1 ``` ## EGRESS SCHEDULE <table><tr><th>EXIT</th><th>AREA SERVED</th><th>OCC. LOAD</th><th>REQ'D CLR WIDTH (IN)</th><th>PROVIDED (IN)</th></tr><tr><td>EXIT 1</td><td>MAIN HALL"
BRL-PIP-ISO-0314,"<!-- page_num: 0 --> <table><tr><th>LINE NO.</th><th>SERVICE</th><th>NPS</th><th>CLASS</th><th>MATERIAL</th><th>DESIGN PRESS</th><th>DESIGN TEMP</th><th>FLANGE CLASS</th></tr><tr><td>6""-BOG-1851-C6S</td><td>BOG</td><td>6</td><td>C6S</td><td>A312 TP316L</td><td>1270 PSIG</td><td>-270 / 140 F</td><td>CL600 RF</td></tr></table> ```figure N W5 FW VT 3'-7"" 11'-2"" FROM V-1076 NOZZLE N3 FLG 6"" CL600 RF WN W4 BW RT 100% TO E-1161 NOZZLE N2 FLG 6"" CL600 RF WN 10'-4"" FLG 6"" CL600 RF WN 3'-0"" FLG 6"" CL60"
BRL-PIP-ISO-0119,"<!-- page_num: 0 --> <table><tr><th>LINE NO.</th><th>SERVICE</th><th>NPS</th><th>CLASS</th><th>MATERIAL</th><th>DESIGN PRESS</th><th>DESIGN TEMP</th><th>FLANGE CLASS</th></tr><tr><td>4""-LNG-1852-C6S</td><td>LNG</td><td>4</td><td>C6S</td><td>A312 TP316L</td><td>1295 PSIG</td><td>-270 / 140 F</td><td>CL600 RF</td></tr></table> ```figure N W1 BW RT 100% 9'-10"" FLG 4"" CL600 RF WN FROM K-1065 NOZZLE N3 8'-5"" W2 BW RT 100% W4 BW RT 100% 8'-9"" 7'-6"" FLG 4"" CL600 RF WN FLG 4"" CL150 RF WN 5'-11"" W3 BW"
BRL-PIP-PID-1094,"<!-- page_num: 0 --> # PROCESS FLOW AND INSTRUMENTATION - TANK T-203 ```figure PSV-2803 SET 955 PSIG FIC 2860 M FROM LNG RUNDOWN HEADER FLG CL600 M O LOTO-2802A FLG CL600 HV-2871 TO STORAGE TANK 6""-MR-2850-C6S DP 955 PSIG P-2801A LNG TRANSFER PUMP 6""-MR-2850-C6S DP 955 PSIG P-2802A LNG TRANSFER PUMP V-2803 SUCTION DRUM DP 955 PSIG / -270 TO 140 F PSV-2804 SET 1135 PSIG FIC 2861 M O LOTO-2805A FROM FEED GAS HEADER FLG CL600 FLG CL600 HV-2873 TO STORAGE TANK 16""-NG-2851-C6S DP 1135 PSIG M O LOTO"


## Verify

Row counts first: every drawing present once, each with a row in `drawings_raw_tracking`, and how many parsed cleanly.

In [9]:
counts_df = spark.sql(f"""
    SELECT COUNT(*) AS drawings,
           COUNT(DISTINCT drawing_id) AS distinct_ids,
           COUNT(IF(parse_status = 'ok', 1, NULL)) AS ok_count,
           COUNT(IF(parse_status <> 'ok', 1, NULL)) AS error_count,
           COUNT(IF(parsed_markdown IS NULL OR parse_json IS NULL, 1, NULL)) AS missing_text
    FROM {TARGET}
""")
display(counts_df)
display(spark.sql(f"SELECT parse_status, COUNT(*) AS drawings FROM {TARGET} GROUP BY parse_status ORDER BY drawings DESC"))

c = counts_df.first()
orphans = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {TARGET} t
    WHERE NOT EXISTS (SELECT 1 FROM {TRACKING} d WHERE d.drawing_id = t.drawing_id AND d.volume_path = t.volume_path)
""").first().n
print(f"{c.drawings} rows, {c.distinct_ids} distinct drawing_id, {orphans} without a drawings_raw_tracking row")
assert c.drawings == n_docs and c.distinct_ids == n_docs and orphans == 0

error_rate = c.error_count / c.drawings
print(f"parse_status ok: {c.ok_count}; error: {c.error_count} ({error_rate:.1%})")
if error_rate > 0.05:
    print("WARNING: more than 5% of the drawings did not parse; look at parse_status before using this table")

drawings,distinct_ids,ok_count,error_count,missing_text
1200,1200,1200,0,0


parse_status,drawings
ok,1200


1200 rows, 1200 distinct drawing_id, 0 without a drawings_raw_tracking row
parse_status ok: 1200; error: 0 (0.0%)


Then whether the text is any good. Two checks that need no ground truth: the parser should return as many pages as the PDF has (the page objects are counted in the PDF bytes; ReportLab writes one `/Type /Page` per sheet), and the drawing number printed in the title block should come back in the text.

In [10]:
pages_df = spark.sql(f"""
    WITH pdf AS (
        SELECT _source_file AS volume_path,
               size(split(decode(content, 'ISO-8859-1'), '/Type /Page[^s]')) - 1 AS pdf_pages
        FROM {BRONZE} WHERE _rescued_data IS NULL
    ), parsed AS (
        SELECT volume_path,
               size(from_json(get_json_object(parse_json, '$.document.pages'), 'ARRAY<STRUCT<id: INT>>')) AS parsed_pages,
               parsed_markdown LIKE CONCAT('%', drawing_id, '%') AS has_own_number
        FROM {TARGET}
    )
    SELECT parsed_pages, pdf_pages, COUNT(*) AS drawings, COUNT_IF(has_own_number) AS text_has_own_drawing_number
    FROM parsed JOIN pdf USING (volume_path)
    GROUP BY parsed_pages, pdf_pages
    ORDER BY parsed_pages, pdf_pages
""")
display(pages_df)
rows = pages_df.collect()
wrong_pages = sum(r.drawings for r in rows if r.parsed_pages != r.pdf_pages)
no_number = sum(r.drawings - r.text_has_own_drawing_number for r in rows)
print(f"{sum(r.parsed_pages * r.drawings for r in rows)} pages parsed; drawings whose parsed page count differs from the PDF's: {wrong_pages}")
print(f"drawings whose text does not contain their own drawing number: {no_number}")

print("\nWhat the parser found, by element type:")
display(spark.sql(f"""
    SELECT e.type AS element_type, COUNT(*) AS elements, COUNT(DISTINCT drawing_id) AS drawings
    FROM (SELECT drawing_id,
                 explode(from_json(get_json_object(parse_json, '$.document.elements'), 'ARRAY<STRUCT<type: STRING>>')) AS e
          FROM {TARGET})
    GROUP BY e.type
    ORDER BY elements DESC
"""))

parsed_pages,pdf_pages,drawings,text_has_own_drawing_number
1,1,1030,1030
2,2,170,170


1370 pages parsed; drawings whose parsed page count differs from the PDF's: 0
drawings whose text does not contain their own drawing number: 0

What the parser found, by element type:


element_type,elements,drawings
text,7390,1200
section_header,6464,1200
table,5917,1200
figure,4780,1065
title,373,333


## One drawing in full

`BRL-INS-PID-0052` is a P&ID with three planted flaws (one each of a missing LOTO tag, a missing relief valve and an under-rated flange; the ground truth lists them with their boxes). First the raw structure in `parse_json`, then the `parsed_markdown` value exactly as stored. The same text is written to `outputs/sample_parsed.md` together with the pixel box of every element, so a reader can see what the parse gives the later tasks.

In [11]:
sample = spark.table(TARGET).where(F.col("drawing_id") == SAMPLE_ID).first()
assert sample is not None and sample.parse_status == "ok", f"{SAMPLE_ID} did not parse"
result = json.loads(sample.parse_json)
pages, elements = result["document"]["pages"], result["document"]["elements"]

print(f"{SAMPLE_ID}: {len(pages)} page, {len(elements)} elements, parse_json {len(sample.parse_json):,} characters, parsed_markdown {len(sample.parsed_markdown):,} characters")
print("parse_json top level:", sorted(result), "| document:", sorted(result["document"]), "| metadata:", json.dumps(result["metadata"]))
print("pages:", json.dumps(pages))
print("one element (the title):", json.dumps(elements[0]))
print("\n----- parsed_markdown -----")
print(sample.parsed_markdown)

BRL-INS-PID-0052: 1 page, 26 elements, parse_json 7,079 characters, parsed_markdown 2,971 characters
parse_json top level: ['document', 'error_status', 'metadata'] | document: ['elements', 'pages'] | metadata: {"file_metadata": null, "id": "3802ab6d-5158-4747-84c5-81d886432739", "version": "2.0"}
pages: [{"id": 0, "image_uri": null}]
one element (the title): {"bbox": [{"coord": [109, 106, 1034, 136], "page_id": 0}], "confidence": 1, "content": "PROCESS FLOW AND INSTRUMENTATION - FLARE AND VENT", "description": null, "id": 0, "type": "title"}

----- parsed_markdown -----
<!-- page_num: 0 -->

# PROCESS FLOW AND INSTRUMENTATION - FLARE AND VENT

```figure
PSV-2503
SET 1890 PSIG
M
O LOTO-2501
FIC
2560
FROM FEED GAS HEADER
FLG CL900
M
FLG CL900
HV-2571
TO STORAGE TANK
6"-NG-2550-D9S
DP 1890 PSIG
K-2501
REFRIGERANT COMPRESSOR
6"-NG-2550-D9S
DP 1890 PSIG
K-2502
BOG COMPRESSOR
V-2503
SEPARATOR
DP 1890 PSIG / -270 TO 140 F
PSV-2504
SET 500 PSIG
FIC
2561
FROM REFRIGERANT HEADER
FLG CL150
M
O LO

Where the planted flaws fall in the parse. The generator's boxes are `[x, y, w, h]` as fractions of the page, origin top-left. `ai_parse_document` boxes are `[x1, y1, x2, y2]` in pixels of the rendered page image. The sheets are 1224 x 792 pt and render at 2.75 px per point (3366 x 2178 px), so dividing a parser box by those two numbers gives the ground truth's fractions. A flaw drawn inside a schematic can only be located as precisely as the element that holds it.

In [12]:
SHEET_PX = (1224 * 2.75, 792 * 2.75)  # width, height of the rendered page image
GROUND_TRUTH = Path("../01_synthetic_data/outputs/ground_truth_findings.jsonl")  # same relative path in the repo and in a bundle deploy
planted = []
if GROUND_TRUTH.exists():
    planted = [f for f in map(json.loads, GROUND_TRUTH.read_text().splitlines()) if f["drawing_id"] == SAMPLE_ID]
print(f"{len(planted)} planted flaws for {SAMPLE_ID} in {GROUND_TRUTH}")


def element_containing(finding):
    # The smallest parsed element on the finding's page whose pixel box contains the finding's box, as (id, type), or None.
    x, y, w, h = finding["bbox"]
    left, top, right, bottom = x * SHEET_PX[0], y * SHEET_PX[1], (x + w) * SHEET_PX[0], (y + h) * SHEET_PX[1]
    hits = []
    for el in elements:
        box = el["bbox"][0]
        x1, y1, x2, y2 = box["coord"]
        if box["page_id"] == finding["page_num"] and x1 <= left and y1 <= top and x2 >= right and y2 >= bottom:
            hits.append(((x2 - x1) * (y2 - y1), el["id"], el["type"]))
    return min(hits)[1:] if hits else None


for f in planted:
    print(f"{f['finding_id']}  {f['flaw_type']:34} page_num {f['page_num']}  -> inside element {element_containing(f)}")

3 planted flaws for BRL-INS-PID-0052 in ../01_synthetic_data/outputs/ground_truth_findings.jsonl
BRL-INS-PID-0052-F01  missing_loto_tag                   page_num 0  -> inside element (1, 'figure')
BRL-INS-PID-0052-F02  missing_pressure_relief_notation   page_num 0  -> inside element (1, 'figure')
BRL-INS-PID-0052-F03  flange_rating_mismatch             page_num 0  -> inside element (1, 'figure')


In [13]:
WHERE_IT_SHOWS = {  # what the text gives a reader, for SAMPLE_ID; phrased by tag number, because label order and spelling vary between parses
    "missing_loto_tag": "the figure text has LOTO tags for `K-2501` and `P-2505A` (`LOTO-2501`, `LOTO-2505A`; the parser sometimes returns the O of LOTO as a zero) but none for compressor `K-2502` (BOG COMPRESSOR)",
    "missing_pressure_relief_notation": "the figure text has vessels `V-2503`, `V-2504` and `V-2506`; the first two come with `PSV-2503 / SET 1890 PSIG` and `PSV-2504 / SET 500 PSIG`, knockout drum `V-2506` has no PSV tag or set pressure",
    "flange_rating_mismatch": "the figure text has a `FLG CL150` among lines tagged `10\"-BOG-2551-B3S` (class code B3S = CL300 in the general notes, `DP 500 PSIG`), and the flange rating table in the same text gives CLASS 150 a maximum of 275 PSIG",
}


def cell(text, width=60):
    text = " / ".join(str(text).split("\n")).replace("|", "\\|")
    return text if len(text) <= width else text[: width - 1] + "…"


lines = [
    f"# Sample: `{SAMPLE_ID}` as parsed by `ai_parse_document`",
    "",
    f"One row of `{TARGET}`, written by [`01_parse_drawings.ipynb`](../01_parse_drawings.ipynb). The text under \"parsed_markdown\" is that column's value, unedited; the element table at the end is read from the row's `parse_json`.",
    "",
    "| column | value |",
    "|---|---|",
    f"| drawing_id | `{sample.drawing_id}` |",
    f"| volume_path | `{sample.volume_path}` |",
    f"| parse_status | `{sample.parse_status}` |",
    f"| parsed_at | {sample.parsed_at:%Y-%m-%d %H:%M:%S} UTC |",
    f"| pages / elements | {len(pages)} / {len(elements)} |",
    f"| size | parsed_markdown {len(sample.parsed_markdown):,} characters, parse_json {len(sample.parse_json):,} characters |",
    "",
    "## Why this drawing",
    "",
    "It is one of the 240 flawed drawings: a P&ID with three planted flaws (`ground_truth_findings.jsonl`). Nothing in the text marks them; they show as missing or inconsistent labels.",
    "",
]
if planted:
    lines += ["| finding_id | flaw_type | citation | severity | page_num | ground-truth bbox [x, y, w, h] | parsed element holding it |", "|---|---|---|---|---|---|---|"]
    for f in planted:
        holder = element_containing(f)
        lines.append(f"| {f['finding_id']} | {f['flaw_type']} | {f['citation']} | {f['severity']} | {f['page_num']} | {f['bbox']} | {'id ' + str(holder[0]) + ' (' + holder[1] + ')' if holder else 'none'} |")
    lines += ["", "What the text gives a reader to work with:", ""]
    lines += [f"- `{f['flaw_type']}`: {WHERE_IT_SHOWS[f['flaw_type']]}." for f in planted if f["flaw_type"] in WHERE_IT_SHOWS]
    lines.append("")
lines += [
    "Parsing is done by a model, so it is not repeatable to the letter: parsing this PDF more than once during the build returned the same labels in a different order, boxes a few pixels apart and a differently spelled LOTO tag. The labels inside a `figure` are one block in the parser's own order, not the layout order, so a flaw inside a schematic cannot be placed more precisely than the figure's box.",
    "",
]
lines += ["## parsed_markdown", "", sample.parsed_markdown, "", "## Elements in parse_json", ""]
lines += [
    "Boxes are `[x1, y1, x2, y2]` in pixels of the rendered page image, origin top-left (3366 x 2178 px for these sheets). `figure` elements also carry a generated `description` in `parse_json`; it is not part of `parsed_markdown`.",
    "",
    "| id | type | page_id | confidence | box (px) | text |",
    "|---|---|---|---|---|---|",
]
for el in elements:
    box = el["bbox"][0]
    lines.append(f"| {el['id']} | {el['type']} | {box['page_id']} | {el['confidence']} | {box['coord']} | {cell(el.get('content') or '')} |")

OUT.mkdir(exist_ok=True)
(OUT / "sample_parsed.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {OUT / 'sample_parsed.md'}: {len(lines)} lines")

wrote outputs/sample_parsed.md: 67 lines


`drawings_parsed` is the input of the next notebooks. `02_extract_classify` reads `parsed_markdown` for `ai_extract` and `ai_classify`; `03_compliance_check` sends the same text to the judge and reads `parse_json` for page numbers and boxes.